In [1]:
import sys
import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.14.0 2.3.4 2.3.3 1.7.2 3.10.7


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 2653         
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


In [4]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]   

In [5]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    mask = np.ones(n_samples, dtype=bool)
    
    indices = np.array(sample_indices, dtype=int)
    
    if len(indices) > 0:
        mask[indices] = False
        
    return mask

In [6]:
print(oob_mask([0, 2, 2], 5))             
print(oob_mask([], 3))                    
print(oob_mask(np.array([3, 0]), 4))      

[False  True False  True  True]
[ True  True  True]
[False  True  True False]


In [7]:
bag.fit(X_train, y_train)

n_train = len(X_train)
oob_shares = []

for k in range(len(bag.estimators_)):
    sample_indices = bag.estimators_samples_[k]
    mask = oob_mask(sample_indices, n_train)
    oob_shares.append(mask.mean()) 

mean_oob_share = np.mean(oob_shares)
mean_inbag_share = 1 - mean_oob_share

formula_oob = (1 - 1/n_train)**n_train
formula_inbag = 1 - formula_oob

print(f"Mean OOB share:   {mean_oob_share:.4f}")
print(f"Mean In-bag share:{mean_inbag_share:.4f}")
print(f"Formula OOB:      {formula_oob:.4f}")
print(f"Formula In-bag:   {formula_inbag:.4f}")

votes = np.zeros((n_train, 2), dtype=int)

for k in range(len(bag.estimators_)):
    sample_indices = bag.estimators_samples_[k]
    mask = oob_mask(sample_indices, n_train)
    
    tree_k = bag.estimators_[k]
    preds = tree_k.predict(X_train[mask])
    
    np.add.at(votes, (np.where(mask)[0], preds), 1)

oob_predictions = np.argmax(votes, axis=1)

manual_oob_accuracy = np.mean(oob_predictions == y_train)

print(f"\nManual OOB accuracy: {manual_oob_accuracy:.4f}")
print(f"sklearn oob_score_:  {bag.oob_score_:.4f}")

Mean OOB share:   0.3677
Mean In-bag share:0.6323
Formula OOB:      0.3678
Formula In-bag:   0.6322

Manual OOB accuracy: 0.9353
sklearn oob_score_:  0.9353


In [8]:
from sklearn.model_selection import cross_validate

models = {
    "tree": tree,
    "bag": bag,
    "rf": rf,
    "lr": lr,
    "bag_lr": bag_lr
}

results = {}

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
    results[name] = scores
    print(f"{name:6s}: mean = {scores.mean():.4f}, std = {scores.std():.4f}")

print("\n--- Paired fold differences (mean ± std) ---")

def print_diff(model_a, model_b):
    diff = results[model_a] - results[model_b]
    print(f"{model_a} - {model_b}: {diff.mean():.4f} ± {diff.std():.4f}")
    print(f"  folds: {np.round(diff, 4)}")

print_diff("bag", "tree")
print_diff("rf", "tree")
print_diff("bag_lr", "lr")

tree  : mean = 0.8938, std = 0.0098
bag   : mean = 0.9343, std = 0.0054
rf    : mean = 0.9451, std = 0.0021
lr    : mean = 0.9378, std = 0.0044
bag_lr: mean = 0.9375, std = 0.0027

--- Paired fold differences (mean ± std) ---
bag - tree: 0.0406 ± 0.0097
  folds: [0.0365 0.0365 0.0539 0.027  0.0492]
rf - tree: 0.0514 ± 0.0107
  folds: [0.0428 0.0444 0.0713 0.0444 0.054 ]
bag_lr - lr: -0.0003 ± 0.0023
  folds: [-0.0016  0.0032  0.0016 -0.0016 -0.0032]


In [9]:
from sklearn.metrics import accuracy_score

rf.fit(X_train, y_train)

oob = rf.oob_score_

cv_mean = results["rf"].mean()
cv_std = results["rf"].std()

p = accuracy_score(y_test, rf.predict(X_test))
n_test = len(y_test)
se = np.sqrt(p * (1 - p) / n_test)

print(f"Random Forest OOB score: {oob:.4f}")
print(f"Random Forest CV mean:   {cv_mean:.4f} (std {cv_std:.4f})")
print(f"Random Forest Test acc:  {p:.4f} ± {se:.4f} (std error)")

Random Forest OOB score: 0.9442
Random Forest CV mean:   0.9451 (std 0.0021)
Random Forest Test acc:  0.9562 ± 0.0063 (std error)


### Part 1: Conclusions

**1. Out-of-bag (OOB) share:**
My OOB share (0.3677) is very close to the formula value (0.3678). My manual OOB accuracy (0.9353) is exactly the same as `bag.oob_score_`. 

**2. Did bagging help?**
- **For the tree:** Yes, it helped a lot. Bagging improved the accuracy by **0.0406 ± 0.0097**. This difference is bigger than the fold spread. It means the improvement is real. Bagging makes a single tree more stable.
- **For logistic regression:** No, it did not help. The difference is **-0.0003 ± 0.0023**. This is inside the fold spread. Logistic regression is already a stable model. So, bagging it does not change the result.

**3. Three estimates for the forest:**
The OOB score (0.9442) is very close to the CV mean (0.9451 ± 0.0021). The difference is smaller than the standard deviation. The test accuracy (0.9562 ± 0.0063) is a little higher. But overall, the OOB score is a good and fair estimate here.

In [10]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, confusion_matrix, precision_score, recall_score

oof_probs = cross_val_predict(rf, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

fpr, tpr, thresholds = roc_curve(y_train, oof_probs, drop_intermediate=False)

valid_idx = np.where(fpr <= 0.01)[0]
best_idx = valid_idx[np.argmax(tpr[valid_idx])]

best_recall = tpr[best_idx]
max_t = np.max(thresholds[valid_idx][tpr[valid_idx] == best_recall])

print(f"Chosen threshold t: {max_t:.4f}")
print(f"OOF FPR at t: {fpr[best_idx]:.4f}")
print(f"OOF Recall at t: {tpr[best_idx]:.4f}")

Chosen threshold t: 0.7500
OOF FPR at t: 0.0095
OOF Recall at t: 0.7963


In [11]:
test_probs = rf.predict_proba(X_test)[:, 1]

y_pred_t = (test_probs >= max_t).astype(int)

cm = confusion_matrix(y_test, y_pred_t)
print("Confusion Matrix:")
print(cm)

precision = precision_score(y_test, y_pred_t)
recall = recall_score(y_test, y_pred_t)

tn, fp, fn, tp = cm.ravel()
test_fpr = fp / (fp + tn)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"FPR: {test_fpr:.4f}")
print(f"Legitimate mails flagged: {fp} out of {fp + tn}")

Confusion Matrix:
[[626   6]
 [ 71 348]]

Precision: 0.9831
Recall: 0.8305
FPR: 0.0095
Legitimate mails flagged: 6 out of 632


In [12]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=SEED, scoring="accuracy")
perm_means = perm.importances_mean

imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Impurity_Importance": rf.feature_importances_,
    "Permutation_Importance": perm_means
})

print("Top 5 by Impurity Importance (rf.feature_importances_):")
display(imp_df.sort_values("Impurity_Importance", ascending=False).head(5))

print("\nTop 5 by Permutation Importance:")
display(imp_df.sort_values("Permutation_Importance", ascending=False).head(5))

Top 5 by Impurity Importance (rf.feature_importances_):


,Feature,Impurity_Importance,Permutation_Importance
51,char_freq_exclam,0.129566,0.019125
52,char_freq_dollar,0.086930,0.014843
6,word_freq_remove,0.074262,0.024167
15,word_freq_free,0.068424,0.006470
54,capital_run_length_average,0.065666,0.013796



Top 5 by Permutation Importance:


,Feature,Impurity_Importance,Permutation_Importance
6,word_freq_remove,0.074262,0.024167
51,char_freq_exclam,0.129566,0.019125
24,word_freq_hp,0.051763,0.018268
52,char_freq_dollar,0.086930,0.014843
54,capital_run_length_average,0.065666,0.013796


### Part 2: Conclusions

**1. Curves:**
The ROC and Precision-Recall curves show that the Random Forest is better than the single tree. The Random Forest has a higher AUC and Average Precision.

**2. Threshold and Filter:**
I found the threshold `t = 0.7500` using only the training data (OOF probabilities). At this threshold, the OOF FPR is 0.0095 (less than 1%). 
On the test data, the filter works well. The Confusion Matrix is:
- TN = 626, FP = 6
- FN = 71, TP = 348

**3. Test Results:**
- **Did the 1% target hold?** Yes. The test FPR is 0.0095 (0.95%). Only 6 legitimate mails were flagged out of 632.
- **Precision:** 0.9831 (when the filter says "spam", it is almost always true).
- **Recall:** 0.8305 (we catch 83% of all spam).

**4. Feature Importances:**
The top features are different. 
- Impurity importance likes `char_freq_exclam` most.
- Permutation importance likes `word_freq_remove` most.
- The feature `word_freq_hp` is in the top 5 for Permutation, but not for Impurity.

**Why do they differ?**
Impurity importance is calculated during training. It shows how often a feature splits the trees. 
Permutation importance is calculated on the test data. It shows how much the accuracy drops if we break one feature. A feature can be great for splitting (high impurity), but another feature can be more important for the final test accuracy (high permutation).

In [13]:
def hard_vote(predictions, weights=None):
    """Weighted majority vote of several models' predicted labels, one label per sample."""
    preds = np.array(predictions)
    n_models, n_samples = preds.shape
    
    if weights is None:
        weights = np.ones(n_models)
    else:
        weights = np.array(weights)
        
    classes = np.unique(preds)
    final_preds = np.zeros(n_samples, dtype=int)
    
    for i in range(n_samples):
        counts = {}
        for m in range(n_models):
            label = preds[m, i]
            counts[label] = counts.get(label, 0) + weights[m]
            
        max_w = -1
        best_label = 0
        for label in sorted(counts.keys()):
            if counts[label] > max_w:
                max_w = counts[label]
                best_label = label
                
        final_preds[i] = best_label
        
    return final_preds

In [14]:
print(hard_vote([[0, 1, 1], [0, 1, 0], [1, 1, 0]]))       # [0 1 0]
print(hard_vote([[0, 1], [1, 0]]))                        # [0 0]
print(hard_vote([[0, 1], [1, 0]], weights=[1, 2]))        # [1 0]
print(hard_vote([[2, 0], [2, 1], [0, 1]]))                # [2 1]

[0 1 0]
[0 0]
[1 0]
[2 1]


In [15]:
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone

fitted_members = [clone(model).fit(X_train, y_train) for name, model in members]

preds_matrix = np.array([m.predict(X_test) for m in fitted_members])

vc_hard = VotingClassifier(members, voting="hard").fit(X_train, y_train)
vc_preds = vc_hard.predict(X_test)

my_preds = hard_vote(preds_matrix)

agreement = np.mean(my_preds == vc_preds)
print(f"Agreement between my hard_vote and sklearn: {agreement:.4f}")

print("\n--- Pairwise Disagreement on Test Set ---")
names = [name for name, model in members]
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        diff = np.mean(preds_matrix[i] != preds_matrix[j])
        print(f"{names[i]} vs {names[j]}: {diff:.4f}")

from sklearn.model_selection import cross_validate

vc_soft = VotingClassifier(members, voting="soft")

cv_results = {}
for name, model in members:
    cv_results[name] = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
cv_results["hard_vote"] = cross_validate(vc_hard, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
cv_results["soft_vote"] = cross_validate(vc_soft, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]

print("\n--- CV Results ---")
for name, scores in cv_results.items():
    print(f"{name:10s}: mean = {scores.mean():.4f}, std = {scores.std():.4f}")

best_member_name = max(names, key=lambda n: cv_results[n].mean())
print(f"\nBest member is: {best_member_name}")

def print_diff_v3(name_a, name_b):
    diff = cv_results[name_a] - cv_results[name_b]
    print(f"{name_a} - {name_b}: {diff.mean():.4f} ± {diff.std():.4f}")

print("\n--- Paired differences ---")
print_diff_v3("hard_vote", best_member_name)
print_diff_v3("soft_vote", best_member_name)

Agreement between my hard_vote and sklearn: 1.0000

--- Pairwise Disagreement on Test Set ---
lr vs knn: 0.0371
lr vs tree: 0.0761
knn vs tree: 0.0885

--- CV Results ---
lr        : mean = 0.9378, std = 0.0044
knn       : mean = 0.9160, std = 0.0100
tree      : mean = 0.8938, std = 0.0098
hard_vote : mean = 0.9407, std = 0.0043
soft_vote : mean = 0.9388, std = 0.0075

Best member is: lr

--- Paired differences ---
hard_vote - lr: 0.0029 ± 0.0025
soft_vote - lr: 0.0010 ± 0.0054


### Part 3: Conclusions

**1. Hard vote check:**
My `hard_vote` function is perfect. It matches the sklearn `VotingClassifier` on 100% of the test data (agreement is 1.0000).

**2. Disagreement:**
The models are quite different. The highest disagreement is between kNN and the tree (0.0885). Logistic Regression (lr) and kNN disagree less (0.0371).

**3. Does voting beat the best member?**
The best member is Logistic Regression (`lr`), with mean accuracy 0.9378.
- **Hard voting** is a little better (0.9407). The paired difference is 0.0029 ± 0.0025. The difference is bigger than the spread. So hard voting helps a little.
- **Soft voting** is not better (0.9388). The paired difference is 0.0010 ± 0.0054. This difference is inside the fold spread. So we cannot say that soft voting is better than `lr`.